# Projected gradient and Frank-Wolfe

Both keep every iterate inside the feasible set, and both learn about that set
from an *oracle* rather than from algebraic constraints:

- `ProjectedGradient` needs a **projection** — the nearest feasible point to a given point
- `FrankWolfe` needs a **linear minimization oracle** — the feasible point minimizing a linear function

MOpt supplies factories that build exact projections for the common sets.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from mopt.nonlinear import (ConstrainedNLPProblem, ProjectedGradient, FrankWolfe,
                            affine_projection, ellipsoid_projection,
                            intersection_projection, ellipsoid_oracle,
                            frank_wolfe_gap)

## Test problem

Minimize a convex quadratic over the intersection of an ellipsoid and a
hyperplane:

$$\min_x\; x^T Q x + b^T x \quad\text{s.t.}\quad x^T M x \le 167,\;\; c^T x = 23.$$

In [2]:
Q = np.array([
    [7.0, -0.2, 0.1, -0.3, 0.0],
    [-0.2, 6.9, 0.4, 0.0, 0.1],
    [0.1, 0.4, 6.6, 0.3, 0.0],
    [-0.3, 0.0, 0.3, 6.7, 0.1],
    [0.0, 0.1, 0.0, 0.1, 6.9],
])
M = np.array([
    [5.0, 0.4, -0.1, 0.1, 0.5],
    [0.4, 6.0, -0.5, -2.5, 1.8],
    [-0.1, -0.5, 4.7, 0.1, -1.5],
    [0.1, -2.5, 0.1, 4.5, -0.3],
    [0.5, 1.8, -1.5, -0.3, 3.7],
])
b = np.array([4.0, -3.0, -3.0, 2.0, 2.0])
c = np.array([-4.0, 1.0, 2.0, 4.0, -5.0])
s_ell, s_eq = 167.0, 23.0

f = lambda x: float(x @ Q @ x + b @ x)
grad_f = lambda x: 2.0 * (Q @ x) + b

P_ellipsoid = ellipsoid_projection(M, s_ell)
P_affine = affine_projection(c[None, :], np.array([s_eq]))
P_both = intersection_projection([P_ellipsoid, P_affine])   # Dykstra

cons = [{"type": "ineq", "fun": lambda y: s_ell - y @ M @ y},
        {"type": "eq",   "fun": lambda y: c @ y - s_eq}]

## Projected gradient

`alpha` sets how far the trial point is thrown before projecting; it is
distinct from the line-search step. Iterates stay feasible throughout, which
the last two columns confirm.

In [3]:
rng = np.random.default_rng(42)
inits = np.vstack([P_both(z) for z in
                   np.vstack([np.zeros(5), rng.uniform(-3, 3, size=(3, 5))])])
alphas = np.round(np.sort(rng.uniform(0.01, 1.0, size=5)), 4)

rows = []
for x0 in inits:
    x_ref = minimize(f, x0, jac=grad_f, method="SLSQP", constraints=cons,
                     options={"maxiter": 500, "ftol": 1e-14}).x
    for alpha in alphas:
        problem = ConstrainedNLPProblem(f=f, x0=x0, grad=grad_f, projection=P_both)
        result = ProjectedGradient(alpha=float(alpha)).solve(problem)
        assert result.success, result.message
        np.testing.assert_allclose(result.x, x_ref, atol=1e-4)
        rows.append({
            "x0": tuple(np.round(x0, 3)),
            "alpha": float(alpha),
            "iters": result.n_iter,
            "f": result.fun,
            "|x - x_scipy|": np.linalg.norm(result.x - x_ref),
            "c^Tx": float(c @ result.x),
            "x^TMx": float(result.x @ M @ result.x),
        })

pd.DataFrame(rows).set_index(["x0", "alpha"])

iters         f  |x - x_scipy|  \
x0                                    alpha                                    
(-1.484, 0.371, 0.742, 1.484, -1.855) 0.0732      6  49.09313   3.974273e-07   
                                      0.2350     35  49.09313   2.308744e-07   
                                      0.5590     28  49.09313   6.940566e-08   
                                      0.6353     29  49.09313   9.445166e-08   
                                      0.8294     40  49.09313   8.193612e-08   
(1.081, -0.226, 2.433, 1.747, -3.139) 0.0732      6  49.09313   1.452053e-07   
                                      0.2350     42  49.09313   2.921888e-07   
                                      0.5590     30  49.09313   9.746840e-08   
                                      0.6353     31  49.09313   1.067298e-07   
                                      0.8294     47  49.09313   7.542471e-08   
(0.476, 2.161, 2.905, 0.146, -3.27)   0.0732      6  49.09313   6.278927e-07   
                                      0.2350     38  49.09313   2.123855e-07   
                                      0.5590     30  49.09313   7.686356e-08   
                                      0.6353     32  49.09313   6.241553e-08   
                                      0.8294     48  49.09313   7.073261e-08   
(-1.173, 2.66, 1.062, 2.335, -0.837)  0.0732      7  49.09313   2.727276e-07   
                                      0.2350     44  49.09313   2.907637e-07   
                                      0.5590     27  49.09313   8.572232e-08   
                                      0.6353     29  49.09313   8.359215e-08   
                                      0.8294     48  49.09313   4.791160e-08   

                                              c^Tx      x^TMx  
x0                                    alpha                    
(-1.484, 0.371, 0.742, 1.484, -1.855) 0.0732  23.0  40.134737  
                                      0.2350  23.0  40.134741  
                                      0.5590  23.0  40.134743  
                                      0.6353  23.0  40.134742  
                                      0.8294  23.0  40.134743  
(1.081, -0.226, 2.433, 1.747, -3.139) 0.0732  23.0  40.134744  
                                      0.2350  23.0  40.134744  
                                      0.5590  23.0  40.134743  
                                      0.6353  23.0  40.134742  
                                      0.8294  23.0  40.134742  
(0.476, 2.161, 2.905, 0.146, -3.27)   0.0732  23.0  40.134751  
                                      0.2350  23.0  40.134743  
                                      0.5590  23.0  40.134743  
                                      0.6353  23.0  40.134742  
                                      0.8294  23.0  40.134742  
(-1.173, 2.66, 1.062, 2.335, -0.837)  0.0732  23.0  40.134738  
                                      0.2350  23.0  40.134741  
                                      0.5590  23.0  40.134743  
                                      0.6353  23.0  40.134743  
                                      0.8294  23.0  40.134742

`alpha` costs iterations, not correctness — every run lands on the same point.
A larger throw means a longer feasible direction each step.

In [4]:
print(pd.DataFrame(rows).groupby("alpha")["iters"].mean().round(1).to_string())

alpha
0.0732     6.2
0.2350    39.8
0.5590    28.8
0.6353    30.2
0.8294    45.8


## Frank-Wolfe on the ellipsoid alone

Frank-Wolfe never projects; it minimizes the linearized objective over the
whole set and steps toward that vertex. It needs a **bounded** set, so drop the
hyperplane and keep the ellipsoid. The Frank-Wolfe gap bounds
$f(x) - f^\star$, so it doubles as a stopping certificate.

In [5]:
lmo = ellipsoid_oracle(M, s_ell)
problem = ConstrainedNLPProblem(f=f, x0=np.zeros(5), grad=grad_f,
                                lmo=lmo, projection=P_ellipsoid)

result = FrankWolfe(max_iter=5000).solve(problem)
x_ref = minimize(f, np.zeros(5), jac=grad_f, method="SLSQP",
                 constraints=[cons[0]], options={"maxiter": 500, "ftol": 1e-14}).x

print(f"success = {result.success}   iters = {result.n_iter}")
print(f"f          = {result.fun:.9f}   (scipy: {f(x_ref):.9f})")
print(f"x^TMx      = {result.x @ M @ result.x:.4f}   (limit {s_ell})")
print(f"FW gap     = {frank_wolfe_gap(problem, result.x):.3e}   (bounds f - f*)")

success = False   iters = 28
f          = -1.534790543   (scipy: -1.534790543)
x^TMx      = 1.2248   (limit 167.0)
FW gap     = 5.019e-04   (bounds f - f*)


## Same problem, either oracle

Supplying both a projection and an LMO lets either solver run on the same
problem object — useful when comparing them.

In [6]:
for name, solver in [("ProjectedGradient", ProjectedGradient(alpha=0.05, max_iter=5000)),
                     ("FrankWolfe", FrankWolfe(max_iter=5000))]:
    r = solver.solve(problem)
    print(f"{name:18} f = {r.fun:.9f}  iters = {r.n_iter:5d}  feasible: "
          f"{r.x @ M @ r.x <= s_ell + 1e-8}")

ProjectedGradient  f = -1.534790543  iters =    13  feasible: True
FrankWolfe         f = -1.534790543  iters =    28  feasible: True
